In [ ]:
!pip install sacrebleu wandb sentencepiece kagglehub

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm
from tqdm.auto import tqdm
import sacrebleu
import wandb
import os
from functools import partial
from dataclasses import dataclass
from torch.cuda.amp import autocast, GradScaler
import kagglehub
import os
import shutil

In [ ]:
def download_dataset_kaggle():
    downloaded_path = kagglehub.dataset_download("nikitasolonitsyn/DL-1-BDZ-2")
    target_path = "./data"
    if not os.path.exists(target_path):
        def ignore_nfs(dir, files):
            return [f for f in files if f.startswith('.nfs')]
        shutil.copytree(downloaded_path, target_path, ignore=ignore_nfs, dirs_exist_ok=True)
    print("Dataset available at:", target_path)
    return target_path

data_folder = download_dataset_kaggle()

In [ ]:
@dataclass
class ModelConfig:
    NUM_HEADS: int = 8
    DIM_MODEL: int = 512
    D_FF: int = 2048
    NUM_ENCODER_LAYERS: int = 6
    NUM_DECODER_LAYERS: int = 6
    DROPOUT: float = 0.1
    VOCAB_SIZE: int = 32000
    PAD_TOKEN_ID: int = 0
    BOS_TOKEN_ID: int = 2
    EOS_TOKEN_ID: int = 3
    MAX_SEQ_LEN: int = 512

@dataclass
class TrainConfig:
    BATCH_SIZE: int = 20
    LR: float = 3e-3
    NUM_EPOCHS: int = 40
    DEVICE: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    DATA_FOLDER: str = "./data"
    LOG_WANDB: bool = False
    USE_BF16: bool = False
    COMPILE: bool = True
    SAVE_PATH: str = "best_model.pt"
    GRAD_ACCUM_STEPS: int = 4

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:

def train_spm(files, model_prefix, vocab_size=32000):
    print("Starte")
    spm.SentencePieceTrainer.train(
        input=','.join(files), model_prefix=model_prefix, vocab_size=vocab_size,
        model_type='bpe', character_coverage=1.0,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        pad_piece='<pad>', unk_piece='<unk>', bos_piece='<bos>', eos_piece='<eos>',
    )
    sp = spm.SentencePieceProcessor()
    sp.load(f'{model_prefix}.model')
    return sp


  Using cached wurlitzer-3.1.1-py3-none-any.whl.metadata (2.5 kB)
Using cached wurlitzer-3.1.1-py3-none-any.whl (8.6 kB)


In [12]:
class TranslationDataset(Dataset):
    def __init__(self, src_sp, tgt_sp, src_file, tgt_file):
        with open(src_file, encoding='utf-8') as f:
            self.src = [l.strip() for l in f]
        with open(tgt_file, encoding='utf-8') as f:
            self.tgt = [l.strip() for l in f]
        self.src_sp, self.tgt_sp = src_sp, tgt_sp

    def __len__(self):
        return len(self.src)

    def __getitem__(self, i):
        src_ids = [self.src_sp.bos_id()] + self.src_sp.encode(self.src[i]) + [self.src_sp.eos_id()]
        tgt_ids = [self.tgt_sp.bos_id()] + self.tgt_sp.encode(self.tgt[i]) + [self.tgt_sp.eos_id()]
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)

def collate(batch, pad_id):
    src, tgt = zip(*batch)
    src = torch.nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=pad_id)
    tgt = torch.nn.utils.rnn.pad_sequence(tgt, batch_first=True, padding_value=pad_id)
    return src, tgt

In [13]:
class TransformerMT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.shared = nn.Embedding(config.VOCAB_SIZE, config.DIM_MODEL)
        self.pos_embedding = nn.Embedding(config.MAX_SEQ_LEN, config.DIM_MODEL)
        self.dropout = nn.Dropout(config.DROPOUT)
        self.transformer = nn.Transformer(
            d_model=config.DIM_MODEL,
            nhead=config.NUM_HEADS,
            num_encoder_layers=config.NUM_ENCODER_LAYERS,
            num_decoder_layers=config.NUM_DECODER_LAYERS,
            dim_feedforward=config.D_FF,
            dropout=config.DROPOUT,
            activation='relu',
            batch_first=True,
            norm_first=True
        )
        self.lm_head = nn.Linear(config.DIM_MODEL, config.VOCAB_SIZE, bias=False)
        self.lm_head.weight = self.shared.weight

    def _shift_right(self, labels):
        shifted = labels.new_zeros(labels.shape)
        shifted[..., 1:] = labels[..., :-1].clone()
        shifted[..., 0] = self.config.BOS_TOKEN_ID
        return shifted

    def _generate_causal_mask(self, size, device):
        return torch.triu(torch.ones(size, size, device=device) * float('-inf'), diagonal=1)

    def forward(self, input_ids, labels=None):
        src_padding_mask = input_ids == self.config.PAD_TOKEN_ID
        src_pos = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        src_emb = self.shared(input_ids) + self.pos_embedding(src_pos)
        src_emb = self.dropout(src_emb)

        if labels is not None:
            tgt_input = self._shift_right(labels)
            tgt_padding_mask = tgt_input == self.config.PAD_TOKEN_ID
            tgt_pos = torch.arange(tgt_input.size(1), device=tgt_input.device).unsqueeze(0)
            tgt_emb = self.shared(tgt_input) + self.pos_embedding(tgt_pos)
            tgt_emb = self.dropout(tgt_emb)

            causal_mask = self._generate_causal_mask(tgt_input.size(1), tgt_input.device)

            output = self.transformer(
                src_emb, tgt_emb,
                tgt_mask=causal_mask,
                src_key_padding_mask=src_padding_mask,
                tgt_key_padding_mask=tgt_padding_mask,
                memory_key_padding_mask=src_padding_mask
            )
            logits = self.lm_head(output)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1),
                                   ignore_index=self.config.PAD_TOKEN_ID)
            return loss, logits
        else:
            memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_padding_mask)
            return memory

    @torch.no_grad()
    def generate(self, input_ids, max_len=100):
        self.eval()
        src_padding_mask = input_ids == self.config.PAD_TOKEN_ID
        src_pos = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        src_emb = self.shared(input_ids) + self.pos_embedding(src_pos)
        src_emb = self.dropout(src_emb)
        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_padding_mask)

        batch_size = input_ids.size(0)
        dec_input = torch.full((batch_size, 1), self.config.BOS_TOKEN_ID, dtype=torch.long, device=input_ids.device)
        finished = torch.zeros(batch_size, dtype=torch.bool, device=input_ids.device)

        for _ in range(max_len):
            tgt_pos = torch.arange(dec_input.size(1), device=dec_input.device).unsqueeze(0)
            tgt_emb = self.shared(dec_input) + self.pos_embedding(tgt_pos)
            tgt_emb = self.dropout(tgt_emb)
            causal_mask = self._generate_causal_mask(dec_input.size(1), dec_input.device)
            tgt_padding_mask = dec_input == self.config.PAD_TOKEN_ID

            output = self.transformer.decoder(
                tgt_emb, memory,
                tgt_mask=causal_mask,
                tgt_key_padding_mask=tgt_padding_mask,
                memory_key_padding_mask=src_padding_mask
            )
            logits = self.lm_head(output[:, -1:, :])
            next_token = logits.argmax(dim=-1)
            dec_input = torch.cat([dec_input, next_token], dim=1)
            finished |= (next_token.squeeze(1) == self.config.EOS_TOKEN_ID)
            if finished.all():
                break
        return dec_input

In [14]:
def train_model(config, train_loader, val_loader, model, src_sp, tgt_sp, val_ref_file):
    device = config.DEVICE
    model.to(device)
    if config.COMPILE and hasattr(torch, 'compile'):
        model = torch.compile(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LR)
    steps_per_epoch = len(train_loader) // config.GRAD_ACCUM_STEPS
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, config.LR,
        epochs=config.NUM_EPOCHS,
        steps_per_epoch=steps_per_epoch
    )

    scaler = torch.amp.GradScaler('cuda', enabled=config.USE_BF16)
    best_bleu = 0.0

    for epoch in range(config.NUM_EPOCHS):
        model.train()
        total_train_loss = 0
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        for step, (src, tgt) in enumerate(pbar, 1):
            print("AAAAAA")
        #     src, tgt = src.to(device), tgt.to(device)

        #     if config.USE_BF16:
        #         with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        #             loss, _ = model(src, labels=tgt)
        #     else:
        #         loss, _ = model(src, labels=tgt)

        #     scaled_loss = loss / config.GRAD_ACCUM_STEPS
        #     scaler.scale(scaled_loss).backward()

        #     if step % config.GRAD_ACCUM_STEPS == 0:
        #         scaler.unscale_(optimizer)
        #         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        #         scaler.step(optimizer)
        #         scaler.update()
        #         scheduler.step()
        #         optimizer.zero_grad()

        #     total_train_loss += loss.item()
        #     pbar.set_postfix(loss=loss.item(), step=step)

        # model.eval()
        # total_val_loss = 0
        # preds = []
        # with torch.no_grad():
        #     for src, tgt in tqdm(val_loader, desc='Validating', leave=False):
        #         src, tgt = src.to(device), tgt.to(device)

        #         if config.USE_BF16:
        #             with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        #                 loss, _ = model(src, labels=tgt)
        #         else:
        #             loss, _ = model(src, labels=tgt)

        #         total_val_loss += loss.item()
        #         out = model.generate(src, max_len=100)
        #         for seq in out:
        #             ids = [i for i in seq.cpu().tolist()
        #                    if i not in (model.config.PAD_TOKEN_ID,
        #                                 model.config.EOS_TOKEN_ID,
        #                                 model.config.BOS_TOKEN_ID)]
        #             preds.append(tgt_sp.decode(ids))

        # avg_train_loss = total_train_loss / len(train_loader)
        # avg_val_loss = total_val_loss / len(val_loader)
        # bleu = sacrebleu.corpus_bleu(preds,
        #     [[line.strip() for line in open(val_ref_file, encoding='utf-8')]]).score

        # print(f'Epoch {epoch+1}: train loss {avg_train_loss:.4f}, '
        #       f'val loss {avg_val_loss:.4f}, BLEU {bleu:.2f}')

        # if bleu > best_bleu:
        #     best_bleu = bleu
        #     state_dict = model.state_dict()
        #     if any(k.startswith('_orig_mod.') for k in state_dict):
        #         new_state_dict = {k.replace('_orig_mod.', ''): v
        #                           for k, v in state_dict.items()}
        #     else:
        #         new_state_dict = state_dict
        #     torch.save(new_state_dict, config.SAVE_PATH)

        # if config.LOG_WANDB:
        #     wandb.log({'train_loss': avg_train_loss,
        #                'val_loss': avg_val_loss,
        #                'bleu': bleu})

In [ ]:
train_cfg = TrainConfig()
model_cfg = ModelConfig()
# os.makedirs(os.path.dirname(train_cfg.SAVE_PATH), exist_ok=True)
src_sp = train_spm([f"{train_cfg.DATA_FOLDER}/train.de-en.de",
                    f"{train_cfg.DATA_FOLDER}/val.de-en.de"], 'spm_de', model_cfg.VOCAB_SIZE)
tgt_sp = train_spm([f"{train_cfg.DATA_FOLDER}/train.de-en.en",
                    f"{train_cfg.DATA_FOLDER}/val.de-en.en"], 'spm_en', model_cfg.VOCAB_SIZE)

model_cfg.VOCAB_SIZE = max(src_sp.vocab_size(), tgt_sp.vocab_size())
model_cfg.PAD_TOKEN_ID = src_sp.pad_id()
model_cfg.BOS_TOKEN_ID = src_sp.bos_id()
model_cfg.EOS_TOKEN_ID = src_sp.eos_id()

train_ds = TranslationDataset(src_sp, tgt_sp,
                                f"{train_cfg.DATA_FOLDER}/train.de-en.de",
                                f"{train_cfg.DATA_FOLDER}/train.de-en.en")
val_ds = TranslationDataset(src_sp, tgt_sp,
                              f"{train_cfg.DATA_FOLDER}/val.de-en.de",
                              f"{train_cfg.DATA_FOLDER}/val.de-en.en")

collate_fn = partial(collate, pad_id=model_cfg.PAD_TOKEN_ID)
train_loader = DataLoader(train_ds, batch_size=train_cfg.BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=train_cfg.BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, pin_memory=True)

model = TransformerMT(model_cfg)

if train_cfg.LOG_WANDB:
    wandb.init(project='translation-minimal', config={**train_cfg.__dict__, **model_cfg.__dict__})

train_model(train_cfg, train_loader, val_loader, model, src_sp, tgt_sp,
            f"{train_cfg.DATA_FOLDER}/val.de-en.en")

if train_cfg.LOG_WANDB:
    wandb.finish()


In [ ]:
@dataclass
class InferenceConfig:
    MODEL_PATH: str = "/content/drive/MyDrive/translation_model/best_model.pt"
    TEST_FILE: str = "/content/data/data/test1.de-en.de"
    OUTPUT_FILE: str = "/content/drive/MyDrive/translation_model/translations.txt"
    BATCH_SIZE: int = 32
    MAX_LEN: int = 100

In [ ]:
def inference(inf_config: InferenceConfig):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    src_sp = spm.SentencePieceProcessor()
    src_sp.load('spm_de.model')
    tgt_sp = spm.SentencePieceProcessor()
    tgt_sp.load('spm_en.model')
    config = ModelConfig()
    config.VOCAB_SIZE = max(src_sp.vocab_size(), tgt_sp.vocab_size())
    config.PAD_TOKEN_ID = src_sp.pad_id()
    config.BOS_TOKEN_ID = src_sp.bos_id()
    config.EOS_TOKEN_ID = src_sp.eos_id()

    model = TransformerMT(config)
    state_dict = torch.load(inf_config.MODEL_PATH, map_location='cpu')
    # Remove '_orig_mod.' prefix if present (from torch.compile)
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('_orig_mod.'):
            new_state_dict[k[10:]] = v
        else:
            new_state_dict[k] = v
    model.load_state_dict(new_state_dict)
    model.to(device)
    model.eval()

    test_dataset = TranslationDataset(
        src_sp=src_sp,
        tgt_sp=tgt_sp,
        src_file=inf_config.TEST_FILE,
        tgt_file=inf_config.TEST_FILE
    )
    collate_fn = partial(collate, pad_id=config.PAD_TOKEN_ID)
    test_loader = DataLoader(
        test_dataset,
        batch_size=inf_config.BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True
    )

    all_translations = []
    with torch.no_grad():
        for src, _ in tqdm(test_loader, desc="Translating"):
            src = src.to(device)
            generated = model.generate(src, max_len=inf_config.MAX_LEN)
            for seq in generated:
                ids = [i for i in seq.cpu().tolist() if i not in (config.PAD_TOKEN_ID,
                                                                   config.EOS_TOKEN_ID,
                                                                   config.BOS_TOKEN_ID)]
                all_translations.append(tgt_sp.decode(ids))

    with open(inf_config.OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for line in all_translations:
            f.write(line + '\n')
    print(f"Translations saved to {inf_config.OUTPUT_FILE}")

In [ ]:
inf_cfg = InferenceConfig()
inference(inf_cfg)